<a href="https://colab.research.google.com/github/Angel-ag-1/ML-pipeline/blob/main/work/notebooks/w07_action_playbook_By_Angel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [44]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Angel-ag-1/ML-pipeline.git"
REPO_DIR = "ML-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline


In [45]:
!pip -q install pandas pyarrow huggingface_hub scikit-learn

In [46]:
import pandas as pd

from huggingface_hub import (
    login,
    whoami,
    hf_hub_download
)

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

login(token=HF_TOKEN)

print(
    "Hugging Face account:",
    whoami()["name"]
)

Hugging Face account: angelhi


In [47]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    ),
)

march = pd.read_parquet(march_path)

print("March data shape:", march.shape)

March data shape: (9841378, 30)


In [48]:
march_agg = (
    march
    .groupby(
        ["client_hash_id", "content_hash_id"]
    )
    .agg(
        gsc_impressions=(
            "gsc_impressions",
            "sum"
        ),
        gsc_clicks=(
            "gsc_clicks",
            "sum"
        ),
        gsc_avg_position=(
            "gsc_avg_position",
            "mean"
        ),
        ga4_sessions=(
            "ga4_sessions",
            "sum"
        ),
        ga4_pageviews=(
            "ga4_pageviews",
            "sum"
        )
    )
    .reset_index()
)

march_agg["ctr"] = (
    march_agg["gsc_clicks"]
    /
    march_agg["gsc_impressions"].replace(
        0,
        pd.NA
    )
)

march_agg["ctr"] = (
    march_agg["ctr"].fillna(0)
)

print("Webpages:", len(march_agg))
print("Clients:", march_agg["client_hash_id"].nunique())

march_agg.head()

Webpages: 331437
Clients: 55


/tmp/ipykernel_1531/2384056160.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  march_agg["ctr"].fillna(0)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_pageviews,ctr
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,0.0,0.0
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,0.0,0.0
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,0.0,0.0
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,0.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,0.0,0.0


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks webpages that may need review based on the signals used in the earlier baseline.

Reason codes explain why a page was selected:

- `LOW_VISIBILITY` — little or no search visibility.
- `LOW_CLICKS` — receives impressions but few or no clicks.
- `POOR_POSITION` — appears in lower search positions.

The recommended action is used for prioritisation and should be reviewed by a human before changes are made.

In [49]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("ARCHETYPE → ACTION MAPPING")
print("=" * 70)

archetypes = {
    "No Visibility": {
        "description": "Pages with zero impressions",
        "action": "REVIEW_CONTENT",
        "reason_code": "LOW_VISIBILITY",
        "priority": "High"
    },
    "No Clicks": {
        "description": "Pages with impressions but zero clicks",
        "action": "REVIEW_CONTENT",
        "reason_code": "LOW_CLICKS",
        "priority": "Medium"
    },
    "Poor Position": {
        "description": "Pages ranking beyond position 20",
        "action": "REVIEW_CONTENT",
        "reason_code": "POOR_POSITION",
        "priority": "Medium"
    }
}

for name, config in archetypes.items():
    print(f"\n{name}:")
    print(f"  Description: {config['description']}")
    print(f"  Action: {config['action']}")
    print(f"  Reason Code: {config['reason_code']}")
    print(f"  Priority: {config['priority']}")

print()

# ==============================================================
# BUILD RANKED ACTION QUEUE
# ==============================================================

print("=" * 70)
print("BUILDING RANKED ACTION QUEUE")
print("=" * 70)

queue = march_agg.copy()

# Create the priority score using the Week 4 baseline logic
queue["priority_score"] = (
    (queue["gsc_impressions"] == 0).astype(int) * 3
    +
    (queue["gsc_clicks"] == 0).astype(int) * 2
    +
    (queue["gsc_avg_position"].fillna(100) > 20).astype(int)
)

# Assign a reason code
def get_reason_code(row):
    if row["gsc_impressions"] == 0:
        return "LOW_VISIBILITY"
    elif row["gsc_clicks"] == 0:
        return "LOW_CLICKS"
    elif row["gsc_avg_position"] > 20:
        return "POOR_POSITION"
    else:
        return "NO_ACTION"

queue["reason_code"] = queue.apply(
    get_reason_code,
    axis=1
)

# Add the recommended action
queue["action_label"] = "REVIEW_CONTENT"

# Rank highest-priority pages first
queue = queue.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

print(f"Ranked webpages: {len(queue):,}")

print()

print(
    queue[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "priority_score",
            "reason_code",
            "action_label"
        ]
    ].head(10)
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


ARCHETYPE → ACTION MAPPING

No Visibility:
  Description: Pages with zero impressions
  Action: REVIEW_CONTENT
  Reason Code: LOW_VISIBILITY
  Priority: High

No Clicks:
  Description: Pages with impressions but zero clicks
  Action: REVIEW_CONTENT
  Reason Code: LOW_CLICKS
  Priority: Medium

Poor Position:
  Description: Pages ranking beyond position 20
  Action: REVIEW_CONTENT
  Reason Code: POOR_POSITION
  Priority: Medium

BUILDING RANKED ACTION QUEUE
Ranked webpages: 331,437

            content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  \
0  content_ffa0167618814c34                0           0               NaN   
1  content_004e9c4c32e88631                0           0               NaN   
2  content_0236ef736698e17c                0           0               NaN   
3  content_025f6cfd3c298870                0           0               NaN   
4  content_ff8a67ea33dfebd9                0           0               NaN   
5  content_02752c6c1c60161f                0 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended to help prioritise webpages for manual content review.

It is decision-support only and does not automatically determine that a page should be changed. The results are based on the available March 2026 data and may not apply to future periods.

In [50]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("INTENDED USE AND LIMITS")
print("=" * 70)

print(
    "\nIntended use:"
)

print(
    "- Prioritise webpages for manual review."
)

print(
    "- Support content and SEO decision making."
)

print(
    "- Provide a repeatable ranked queue."
)

print(
    "\nThe playbook does NOT determine:"
)

print(
    "- Whether content is actually poor quality."
)

print(
    "- Whether a page should be deleted."
)

print(
    "- Whether a page should be rewritten automatically."
)

print(
    "- The true cause of low search visibility."
)

print(
    "\nKnown limits:"
)

limits = [
    "Only March 2026 data is used.",
    "Page age is not included.",
    "Indexing status is not included.",
    "Search intent is not included.",
    "Seasonality is not included.",
    "Business value is not included."
]

for limit in limits:
    print("-", limit)

print(
    "\nConclusion:"
)

print(
    "Scores should be treated as prioritisation "
    "signals requiring human review."
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


INTENDED USE AND LIMITS

Intended use:
- Prioritise webpages for manual review.
- Support content and SEO decision making.
- Provide a repeatable ranked queue.

The playbook does NOT determine:
- Whether content is actually poor quality.
- Whether a page should be deleted.
- Whether a page should be rewritten automatically.
- The true cause of low search visibility.

Known limits:
- Only March 2026 data is used.
- Page age is not included.
- Indexing status is not included.
- Search intent is not included.
- Seasonality is not included.
- Business value is not included.

Conclusion:
Scores should be treated as prioritisation signals requiring human review.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A human should review the page, its context, and possible reasons for low performance before taking action.

The system should not automatically publish, rewrite, delete, or make major changes to content.

In [51]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("HUMAN REVIEW RULES")
print("=" * 70)

review_checks = [
    "Is the page intentionally low traffic?",
    "Is the page newly published?",
    "Is search data incomplete?",
    "Could seasonality explain the performance?",
    "Is the page indexed correctly?",
    "Is the page important to the business?",
    "Is the expected value worth the review effort?"
]

for check in review_checks:
    print("CHECK:", check)

print()

print("=" * 70)
print("NO-GO LIST")
print("=" * 70)

no_go_actions = [
    "Do not automatically rewrite content.",
    "Do not automatically delete webpages.",
    "Do not automatically change titles or metadata.",
    "Do not automatically publish changes.",
    "Do not make business decisions using the score alone."
]

for item in no_go_actions:
    print("-", item)

print()

print(
    "✓ Human review is required before "
    "any content action is taken."
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


HUMAN REVIEW RULES
CHECK: Is the page intentionally low traffic?
CHECK: Is the page newly published?
CHECK: Is search data incomplete?
CHECK: Could seasonality explain the performance?
CHECK: Is the page indexed correctly?
CHECK: Is the page important to the business?
CHECK: Is the expected value worth the review effort?

NO-GO LIST
- Do not automatically rewrite content.
- Do not automatically delete webpages.
- Do not automatically change titles or metadata.
- Do not automatically publish changes.
- Do not make business decisions using the score alone.

✓ Human review is required before any content action is taken.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be reviewed if traffic patterns or model performance change.

A retrain or new analysis may be needed when new data becomes available or when the current signals no longer provide useful prioritisation.

In [52]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("MONITORING AND RETRAIN TRIGGERS")
print("=" * 70)

monitoring_metrics = pd.DataFrame({
    "Metric": [
        "Total webpages",
        "High priority pages",
        "High priority percentage",
        "Median impressions",
        "Median clicks",
        "Median average position"
    ],
    "Current March value": [
        len(queue),
        (queue["priority_score"] >= 3).sum(),
        (
            queue["priority_score"] >= 3
        ).mean(),
        queue["gsc_impressions"].median(),
        queue["gsc_clicks"].median(),
        queue["gsc_avg_position"].median()
    ]
})

print(
    monitoring_metrics.to_string(
        index=False
    )
)

print()
print("REVIEW / RETRAIN TRIGGERS:")

triggers = [
    "Large changes in signal distributions.",
    "Large changes in the number of high-priority pages.",
    "Changes in data collection or availability.",
    "New historical data becoming available.",
    "Repeated human-review feedback that recommendations are weak."
]

for trigger in triggers:
    print("-", trigger)

print()

print(
    "Retraining is not automatic. "
    "New data should first be reviewed for quality, "
    "leakage, and validation design."
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


MONITORING AND RETRAIN TRIGGERS
                  Metric  Current March value
          Total webpages        331437.000000
     High priority pages        188717.000000
High priority percentage             0.569390
      Median impressions             2.000000
           Median clicks             0.000000
 Median average position             8.505296

REVIEW / RETRAIN TRIGGERS:
- Large changes in signal distributions.
- Large changes in the number of high-priority pages.
- Changes in data collection or availability.
- New historical data becoming available.
- Repeated human-review feedback that recommendations are weak.

Retraining is not automatic. New data should first be reviewed for quality, leakage, and validation design.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported to `work/outputs/`.

This exported file can be used as supporting material for the recommendations section of the research paper.

In [53]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("EXPORTS FOR THE PAPER")
print("=" * 70)

import os

# Create the output folder if it does not exist
os.makedirs(
    "work/outputs",
    exist_ok=True
)

# Select the columns for the action playbook queue
export_queue = queue[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_pageviews",
        "ctr",
        "priority_score",
        "reason_code",
        "action_label"
    ]
].copy()

# Export the ranked queue
action_playbook_path = (
    "work/outputs/action_playbook_queue.csv"
)

export_queue.to_csv(
    action_playbook_path,
    index=False
)

print(
    f"✓ Action playbook queue saved: "
    f"{action_playbook_path}"
)

print(
    f"✓ Rows exported: "
    f"{len(export_queue):,}"
)

print()

print("=" * 70)
print("EXPORTS VERIFICATION")
print("=" * 70)

exports = [
    "work/outputs/baseline_action_score.csv",
    "work/outputs/action_playbook_queue.csv"
]

for file_path in exports:
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)

        print(
            f"✓ {file_path} "
            f"({file_size:,} bytes)"
        )
    else:
        print(
            f"✗ {file_path} NOT FOUND"
        )

print()
print("✓ Export check complete.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


EXPORTS FOR THE PAPER
✓ Action playbook queue saved: work/outputs/action_playbook_queue.csv
✓ Rows exported: 331,437

EXPORTS VERIFICATION
✗ work/outputs/baseline_action_score.csv NOT FOUND
✓ work/outputs/action_playbook_queue.csv (35,914,462 bytes)

✓ Export check complete.
